# GEDI Canopy Height Pipeline for Rivanna Watershed

This notebook processes NASA GEDI Level 2A HDF5 files stored in S3 through four sequential phases:

| Phase | Description |
|---|---|
| **Phase 1** | High-performance batch extraction of GEDI shots from S3 |
| **Phase 1a** | Year parsing from GEDI filenames |
| **Phase 1b** | Harmonization to the SMAP 9 km grid |
| **Phase 1c** | Spatial join to assign shots to study-area jurisdictions |

**Source:** `s3://central-virginia-tree-canopy-project/GEDI/GEDI02_A/002/`  
**Final summary output:** `s3://central-virginia-tree-canopy-project/gedi-county-summary/virginia_gedi_county_summary-rvwatershed.csv`
**Final detailed output:** `s3://central-virginia-tree-canopy-project/gedi-county-detailed/virginia_gedi_county_detailed-rvwatershed.csv`

## Install Dependencies

In [1]:
import subprocess, sys

# Pin the entire boto stack atomically — all four packages must be compatible
boto_stack = [
    "aiobotocore==3.7.0",
    "botocore==1.43.0",
    "boto3==1.43.0",
    "s3fs==2026.6.0",
]

# Install boto stack with --no-deps to prevent pip from overriding the pins
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       "--force-reinstall"] + boto_stack)

# Install all other dependencies normally
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       "aioitertools", "h5py", "geopandas", "xarray", "netCDF4", "pyarrow"])

print("All dependencies installed.")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
awscli 1.45.39 requires botocore==1.43.39, but you have botocore 1.43.0 which is incompatible.
awscli 1.45.39 requires s3transfer<0.20.0,>=0.19.0, but you have s3transfer 0.17.1 which is incompatible.
sagemaker 2.257.3 requires attrs<26,>=24, but you have attrs 26.1.0 which is incompatible.


All dependencies installed.


## Imports

In [2]:
import os
import time
import io
import re
import urllib.request

from concurrent.futures import ThreadPoolExecutor, as_completed

import statistics

import boto3
import s3fs
import h5py
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr

print("Imports successful.")

Imports successful.


## Configuration
Edit this cell to change S3 paths, bounding box, grid resolution, or target jurisdictions.

In [3]:
# ── S3 settings ───────────────────────────────────────────────────────────────
S3_BUCKET             = "central-virginia-tree-canopy-project"
S3_PREFIX             = "GEDI/GEDI02_A/002/"     
GEDI_COUNTY_S3_PREFIX = "gedi-county-summary/"
GEDI_DETAILED_COUNTY_S3_PREFIX = "gedi-county-detailed/"
GEDI_INFO_KEY = "data/gedi-folder/rivanna-watershed/gedi02_A_forRivannaWatershed.txt"

# ── Local output directory ────────────────────────────────────────────────────
OUTPUT_DIR        = "/home/ec2-user/SageMaker/gedi_output"
OUTPUT_PARQUET    = os.path.join(OUTPUT_DIR, "virginia_gedi_canopy_multiyear-rvwatershed.parquet")
OUTPUT_NETCDF     = os.path.join(OUTPUT_DIR, "virginia_gedi_canopy_grid-rvwatershed.nc")
OUTPUT_COUNTY_CSV = os.path.join(OUTPUT_DIR, "virginia_gedi_county_summary-rvwatershed.csv")
OUTPUT_DETAILED_COUNTY_CSV = os.path.join(OUTPUT_DIR, "virginia_gedi_county_detailed-rvwatershed.csv")
OUTPUT_DETAILED_CANOPY_HEIGHT_CSV = os.path.join(OUTPUT_DIR, "virginia_gedi_canopy_height-rvwatershed.csv")
OUTPUT_COUNTY_CANOPY_HEIGHT_CSV = os.path.join(OUTPUT_DIR, "virginia_gedi_county_canopy_height-rvwatershed.csv")

# ── Spatial bounds (Padded Rivanna Watershed jurisdiction study area) ───────────────────────
MIN_LON, MIN_LAT, MAX_LON, MAX_LAT = -78.8892, 37.6405, -77.6373, 38.5255

# ── SMAP grid resolution (~9 km) ─────────────────────────────────────────────
GRID_RES = 0.081

# ── Target jurisdictions (Name, State FIPS, County/Place FIPS, Type) ─────────
TARGET_JURISDICTIONS = [
    ("Albemarle",       "51", "003",   "county"),
    ("Augusta",         "51", "015",   "county"),
    ("Buckingham",      "51", "029",   "county"),
    ("Charlottesville", "51", "14968", "place"),
    ("Fluvanna",        "51", "065",   "county"),
    ("Greene",          "51", "079",   "county"),
    ("Louisa",          "51", "109",   "county"),
    ("Nelson",          "51", "125",   "county"),
    ("Rockingham",      "51", "165",   "county"),
]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configuration loaded.")
print(f"  GEDI source  : s3://{S3_BUCKET}/{S3_PREFIX}")
print(f"  Output dir   : {OUTPUT_DIR}")
print(f"  S3 output    : s3://{S3_BUCKET}/{GEDI_COUNTY_S3_PREFIX}")

Configuration loaded.
  GEDI source  : s3://central-virginia-tree-canopy-project/GEDI/GEDI02_A/002/
  Output dir   : /home/ec2-user/SageMaker/gedi_output
  S3 output    : s3://central-virginia-tree-canopy-project/gedi-county-summary/


## Helper Functions

In [4]:
def parse_year_from_filename(filename: str):
    """Extract the year from a standard GEDI filename (e.g., GEDI02_A_2022143...)."""
    year_match = re.search(r'GEDI02_A_(\d{4})', filename)
    if year_match:
        return int(year_match.group(1))
    return None


def fetch_boundary(name: str, state_fips: str, geo_id: str, geo_type: str) -> gpd.GeoDataFrame:
    """Fetch boundary GeoJSON directly from the US Census TIGERweb API."""
    if geo_type == "place":
        url = (
            "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/"
            "Places_CouSub_ConCity_SubMCD/MapServer/4/query"
            f"?where=STATE='{state_fips}'+AND+PLACE='{geo_id}'"
            "&outFields=NAME,STATE,PLACE&f=geojson&outSR=4326"
        )
    else:
        url = (
            "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/"
            "State_County/MapServer/11/query"
            f"?where=STATE='{state_fips}'+AND+COUNTY='{geo_id}'"
            "&outFields=NAME,STATE,COUNTY&f=geojson&outSR=4326"
        )
    print(f"  Fetching boundary for {name}...")
    with urllib.request.urlopen(url, timeout=20) as r:
        gdf = gpd.read_file(r)
    if gdf.empty:
        raise ValueError(f"No boundary found for {name} (FIPS {state_fips}/{geo_id})")
    gdf = gdf.set_crs("EPSG:4326")
    gdf['jurisdiction'] = name
    return gdf

def extract_h5_filename(url: str) -> str:
    """
    Extract just the .h5 file name (basename) from a full granule URL.
 
    Parameters
    ----------
    url : str
        Full URL to a .h5 granule, e.g.
        https://.../GEDI02_A_2025189013841_..._V002/GEDI02_A_2025189013841_..._V002.h5
 
    Returns
    -------
    str
        Just the filename, e.g. GEDI02_A_2025189013841_..._V002.h5
    """
    path = urlparse(url).path
    return posixpath.basename(path)
 
 
def read_url_list_from_s3(bucket: str, key: str, profile: str = None) -> list[str]:
    """
    Fetch a text object from S3 and return the .h5 filename from each non-empty line.
 
    Parameters
    ----------
    bucket : str
        S3 bucket name.
    key : str
        S3 object key (path within the bucket).
    profile : str, optional
        Named AWS CLI profile to use. If None, uses the default credential chain.
 
    Returns
    -------
    list[str]
        List of .h5 file names, one per non-blank line in the source file.
    """
    session = boto3.Session(profile_name=profile) if profile else boto3.Session()
    s3 = session.client("s3")
 
    try:
        #log.info("Fetching s3://%s/%s ...", bucket, key)
        print(f"Fetching s3://{bucket}/{key} ...")
        response = s3.get_object(Bucket=bucket, Key=key)
        body = response["Body"].read().decode("utf-8")
    except NoCredentialsError:
        #log.error("No AWS credentials found. Configure ~/.aws/credentials, "
        #           "set AWS_PROFILE, or pass --profile.")
        print("No AWS credentials found. Configure ~/.aws/credentials, set AWS_PROFILE, or pass --profile.")

        raise
    except ClientError as e:
        error_code = e.response.get("Error", {}).get("Code", "Unknown")
        if error_code == "NoSuchKey":
            #log.error("Object not found: s3://%s/%s", bucket, key)
            print(f"Object not found: s3://{bucket}/{key}")
        elif error_code in ("AccessDenied", "403"):
            print("TEST")
        else:
            #log.error("S3 error (%s) reading s3://%s/%s: %s", error_code, bucket, key, e)
            print(f"S3 error ({error_code}) reading s3://{bucket}/{key}: {e}")
        raise
 
    lines = [line.strip() for line in body.splitlines() if line.strip()]
    filenames = [extract_h5_filename(line) for line in lines]
    #log.info("Read %d line(s) from s3://%s/%s and extracted %d .h5 filename(s)",
    #          len(lines), bucket, key, len(filenames))
    print(f"Read {len(lines)} line(s) from s3://{bucket}/{key} and extracted {len(filenames)} .h5 filename(s)")
    return filenames

# This update works

def process_one_file(key):
    """
    Download one GEDI .h5 granule from S3 into memory (single bulk transfer,
    not lazy range-requests -- HDF5's internal metadata reads make lazy
    access over a network filesystem far slower due to per-request latency),
    then apply spatial + quality filtering. Returns a list of per-beam
    DataFrames (possibly empty).
    """
    file_name = os.path.basename(key)
    results = []

    year = parse_year_from_filename(file_name)
    if not year:
        print(f"Warning: Could not parse year from {file_name}. Skipping.")
        return results

    try:
        s3_path = f"s3://{S3_BUCKET}/{key}"
        with fs.open(s3_path, "rb") as f:
            raw_bytes = f.read()

        with h5py.File(io.BytesIO(raw_bytes), "r") as hf:
            beams = [k for k in hf.keys() if k.startswith("BEAM")]
            for beam in beams:
                if "lat_lowestmode" not in hf[beam]:
                    continue

                lats = hf[f"{beam}/lat_lowestmode"][:]
                lons = hf[f"{beam}/lon_lowestmode"][:]
                spatial_mask = (
                    (lons >= MIN_LON) & (lons <= MAX_LON) &
                    (lats >= MIN_LAT) & (lats <= MAX_LAT)
                )
                if not np.any(spatial_mask):
                    continue

                quality     = hf[f"{beam}/quality_flag"][:][spatial_mask]
                sensitivity = hf[f"{beam}/sensitivity"][:][spatial_mask]
                rh98        = hf[f"{beam}/rh"][:, 98][spatial_mask]

                beam_df = pd.DataFrame({
                    "longitude":    lons[spatial_mask],
                    "latitude":     lats[spatial_mask],
                    "quality_flag": quality,
                    "sensitivity":  sensitivity,
                    "rh98":         rh98,
                    "year":         year,
                    "file_source":  file_name,
                    "beam":         beam
                })

                valid_df = beam_df[
                    (beam_df["quality_flag"] == 1) &
                    (beam_df["sensitivity"]  > 0.9) &
                    (beam_df["rh98"]         > 0)   &
                    (beam_df["rh98"]         < 100)
                ]
                if not valid_df.empty:
                    results.append(valid_df)

    except Exception as e:
        print(f"Error reading {file_name}: {e}")

    return results

print("Helper functions defined.")

Helper functions defined.


In [12]:
from urllib.parse import urlparse
import posixpath

from botocore.exceptions import ClientError

s3_client = boto3.client('s3')

county_specific_gedi_files = []
missing_files = []

filenames = read_url_list_from_s3(S3_BUCKET, GEDI_INFO_KEY, profile=None)

print(f"\n{len(filenames)} GEDI .h5 file name(s):\n")

for name in filenames:
    county_specific_gedi_files.append(name)
    #Does the filename exist in our data repository?
    #print("Checking file existence in S3...")
    # Construct the full S3 key path for the file
    s3_key = f"{S3_PREFIX}{name}"
    try:
        # head_object returns metadata if the file exists
        s3_client.head_object(Bucket=S3_BUCKET, Key=s3_key)
        #existing_files.append(filename)
        #print(f"[EXISTS] s3://{S3_BUCKET}/{s3_key}")
        
    except ClientError as e:
        # An error response code of 404 means the file does not exist
        if e.response['Error']['Code'] == "404":
            missing_files.append(name)
            print(f"[MISSING] s3://{S3_BUCKET}/{s3_key}")
        else:
            # Handle other AWS permissions or network errors (e.g., 403 Forbidden)
            print(f"[ERROR] Could not check {name}: {e.response['Error']['Message']}")
    
    #print(name)

print(f"Found {len(county_specific_gedi_files)} GEDI County Specific HDF5 files.")
if county_specific_gedi_files:
    print(f"  First : {county_specific_gedi_files[0]}")
    print(f"  Last  : {county_specific_gedi_files[-1]}")


Fetching s3://central-virginia-tree-canopy-project/data/gedi-folder/rivanna-watershed/gedi02_A_forRivannaWatershed.txt ...
Read 317 line(s) from s3://central-virginia-tree-canopy-project/data/gedi-folder/rivanna-watershed/gedi02_A_forRivannaWatershed.txt and extracted 317 .h5 filename(s)

317 GEDI .h5 file name(s):

Found 317 GEDI County Specific HDF5 files.
  First : GEDI02_A_2025189013841_O37209_02_T08829_02_004_02_V002.h5
  Last  : GEDI02_A_2019113174925_O02048_03_T02621_02_003_01_V002.h5


## Phase 1 & 1a: Discover GEDI Files in S3

In [13]:
print(f"Scanning s3://{S3_BUCKET}/{S3_PREFIX} for GEDI HDF5 files...Only use county specific files.")

s3 = boto3.client("s3")
paginator = s3.get_paginator("list_objects_v2")

h5_keys = []
for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=S3_PREFIX):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".h5"):
            #Check for a match in the county_specific_gedi_files array
            target_filename = posixpath.basename(obj["Key"])
            if target_filename in county_specific_gedi_files:
                #Add target filename to h5_keys array
                h5_keys.append(obj["Key"])

print(f"Found {len(h5_keys)} GEDI HDF5 files.")
if h5_keys:
    print(f"  First : {h5_keys[0]}")
    print(f"  Last  : {h5_keys[-1]}")

Scanning s3://central-virginia-tree-canopy-project/GEDI/GEDI02_A/002/ for GEDI HDF5 files...Only use county specific files.
Found 317 GEDI HDF5 files.
  First : GEDI/GEDI02_A/002/GEDI02_A_2019113174925_O02048_03_T02621_02_003_01_V002.h5
  Last  : GEDI/GEDI02_A/002/GEDI02_A_2025189013841_O37209_02_T08829_02_004_02_V002.h5


## Phase 1 & 1a: Batch Extraction with Spatial Masking and Quality Filtering

In [14]:
# Batch extraction helper function

def extract_gedi_dataframes(h5_keys, max_workers=16):
    """
    Concurrently download and process GEDI .h5 granules from S3, returning
    a combined list of per-beam DataFrames that passed spatial/quality filtering.

    Parameters
    ----------
    h5_keys : list[str]
        S3 object keys for the .h5 granules to process.
    max_workers : int, optional
        Number of concurrent threads to use for downloading/processing files
        (default: 16). Network I/O bound, so pushing this higher is usually
        safe until you hit S3 throttling or bandwidth limits.

    Returns
    -------
    list[pd.DataFrame]
        List of per-beam DataFrames, one per beam that had valid data after
        spatial masking and quality filtering.
    """
    all_dfs = []
    cell_start_time = time.time()
    completed = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_one_file, key): key for key in h5_keys}
        for future in as_completed(futures):
            key = futures[future]
            try:
                file_dfs = future.result()
                all_dfs.extend(file_dfs)
            except Exception as e:
                print(f"Unhandled error processing {os.path.basename(key)}: {e}")
            completed += 1
            if completed % 10 == 0:
                elapsed = time.time() - cell_start_time
                print(f"Processed {completed}/{len(h5_keys)} files... "
                      f"({elapsed:.2f}s elapsed total)")

    total_elapsed = time.time() - cell_start_time
    minutes, seconds = divmod(total_elapsed, 60)
    print(f"\nExtraction complete. Beams with valid data: {len(all_dfs)}")
    print(f"🚀 Total execution time: {int(minutes)}m {seconds:.2f}s")

    return all_dfs

print("Batch extraction helper function defined.")

Batch extraction helper function defined.


In [15]:
fs = s3fs.S3FileSystem(anon=False)

all_dfs = extract_gedi_dataframes(h5_keys, max_workers=9)  #Max Workers can be set to 8,16 or 24 


Processed 10/317 files... (124.12s elapsed total)
Processed 20/317 files... (238.48s elapsed total)
Processed 30/317 files... (380.72s elapsed total)
Processed 40/317 files... (499.80s elapsed total)
Processed 50/317 files... (630.79s elapsed total)
Processed 60/317 files... (773.85s elapsed total)
Processed 70/317 files... (893.97s elapsed total)
Processed 80/317 files... (1002.08s elapsed total)
Processed 90/317 files... (1162.29s elapsed total)
Processed 100/317 files... (1299.35s elapsed total)
Processed 110/317 files... (1393.49s elapsed total)
Processed 120/317 files... (1539.72s elapsed total)
Processed 130/317 files... (1667.65s elapsed total)
Processed 140/317 files... (1780.04s elapsed total)
Processed 150/317 files... (1907.29s elapsed total)
Processed 160/317 files... (2031.09s elapsed total)
Processed 170/317 files... (2146.39s elapsed total)
Processed 180/317 files... (2260.33s elapsed total)
Processed 190/317 files... (2385.08s elapsed total)
Processed 200/317 files... (

## Save Extracted Points to Parquet

In [16]:
if not all_dfs:
    raise RuntimeError("No valid GEDI shots found within the bounding box. Check S3 prefix and bbox settings.")

df_gedi_rvwatershed = pd.concat(all_dfs, ignore_index=True)
df_gedi_rvwatershed.to_parquet(OUTPUT_PARQUET, index=False)

print(f"Total valid GEDI shots saved : {len(df_gedi_rvwatershed):,}")
print(f"Years covered                : {sorted(df_gedi_rvwatershed['year'].unique())}")
print(f"Parquet file                 : {OUTPUT_PARQUET}")
df_gedi_rvwatershed.head()

Total valid GEDI shots saved : 1,435,350
Years covered                : [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Parquet file                 : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_canopy_multiyear-rvwatershed.parquet


,longitude,latitude,quality_flag,sensitivity,rh98,year,file_source,beam
0,-77.763057,38.504960,1,0.917028,13.550000,2019,GEDI02_A_2019138075448_O02430_03_T00999_02_003...,BEAM0000
1,-77.762549,38.504632,1,0.919079,23.889999,2019,GEDI02_A_2019138075448_O02430_03_T00999_02_003...,BEAM0000
2,-77.762046,38.504307,1,0.919063,14.640000,2019,GEDI02_A_2019138075448_O02430_03_T00999_02_003...,BEAM0000
3,-77.761541,38.503980,1,0.944750,11.230000,2019,GEDI02_A_2019138075448_O02430_03_T00999_02_003...,BEAM0000
4,-77.761033,38.503652,1,0.935242,16.400000,2019,GEDI02_A_2019138075448_O02430_03_T00999_02_003...,BEAM0000


## Save Extracted data points to CSV locally to support Canopy Height Change statistics

In [17]:
#Save to CSV locally to support Canopy Height Change
df_gedi_rvwatershed.to_csv(OUTPUT_DETAILED_CANOPY_HEIGHT_CSV, index=False)
print(f"Saved locally to : {OUTPUT_DETAILED_CANOPY_HEIGHT_CSV}")

Saved locally to : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_canopy_height-rvwatershed.csv


## Phase 1b: Harmonize to the SMAP 9 km Grid

In [18]:
lon_bins = np.arange(MIN_LON, MAX_LON, GRID_RES)
lat_bins = np.arange(MIN_LAT, MAX_LAT, GRID_RES)

df_gedi_rvwatershed['lon_grid'] = pd.cut(df_gedi_rvwatershed['longitude'], bins=lon_bins, labels=lon_bins[:-1]).astype(float)
df_gedi_rvwatershed['lat_grid'] = pd.cut(df_gedi_rvwatershed['latitude'],  bins=lat_bins, labels=lat_bins[:-1]).astype(float)

gedi_grid_rvwatershed = df_gedi_rvwatershed.groupby(['year', 'lat_grid', 'lon_grid'])['rh98'].mean().reset_index()

ds_gedi = gedi_grid_rvwatershed.set_index(['year', 'lat_grid', 'lon_grid']).to_xarray()
ds_gedi.to_netcdf(OUTPUT_NETCDF)

print(f"Grid cells produced for the City of Charlottesville : {len(gedi_grid_rvwatershed):,}")
print(f"NetCDF saved to     : {OUTPUT_NETCDF}")
gedi_grid_rvwatershed.head()

Grid cells produced for the City of Charlottesville : 976
NetCDF saved to     : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_canopy_grid-rvwatershed.nc


,year,lat_grid,lon_grid,rh98
0,2019,37.6405,-78.8892,20.900103
1,2019,37.6405,-78.8082,21.701555
2,2019,37.6405,-78.7272,17.496504
3,2019,37.6405,-78.6462,16.963825
4,2019,37.6405,-78.5652,16.039221


## Phase 1c: Fetch Jurisdiction Boundaries from US Census TIGERweb

In [25]:
boundary_gdfs = []

for name, st_fips, geo_id, geo_type in TARGET_JURISDICTIONS:
    try:
        b_gdf = fetch_boundary(name, st_fips, geo_id, geo_type)
        boundary_gdfs.append(b_gdf)
    except Exception as e:
        print(f"  Failed to fetch boundary for {name}: {e}")

if not boundary_gdfs:
    raise RuntimeError("No jurisdiction boundaries fetched. Cannot perform spatial join.")

filtered_counties = pd.concat(boundary_gdfs, ignore_index=True)
print(f"\nBoundaries fetched: {len(filtered_counties)} jurisdiction(s)")
filtered_counties[['jurisdiction', 'geometry']].head(10)

  Fetching boundary for Albemarle...
  Fetching boundary for Augusta...
  Fetching boundary for Buckingham...
  Fetching boundary for Charlottesville...
  Fetching boundary for Fluvanna...
  Fetching boundary for Greene...
  Fetching boundary for Louisa...
  Fetching boundary for Nelson...
  Fetching boundary for Rockingham...

Boundaries fetched: 9 jurisdiction(s)


,jurisdiction,geometry
0,Albemarle,"POLYGON ((-78.64831 37.73272, -78.64824 37.732..."
1,Augusta,"POLYGON ((-78.93069 38.31342, -78.93242 38.314..."
2,Buckingham,"POLYGON ((-78.24812 37.64089, -78.248 37.64158..."
3,Charlottesville,"POLYGON ((-78.52368 38.02233, -78.52371 38.022..."
4,Fluvanna,"POLYGON ((-78.15905 37.74933, -78.15889 37.749..."
5,Greene,"POLYGON ((-78.3587 38.29019, -78.35879 38.2904..."
6,Louisa,"POLYGON ((-78.30676 38.00647, -78.30687 38.006..."
7,Nelson,"POLYGON ((-78.90797 37.97698, -78.90798 37.976..."
8,Rockingham,"POLYGON ((-79.09295 38.66441, -79.09295 38.664..."


## 10 — Phase 1c: Spatial Join and County-Level Aggregation

In [26]:
gdf_gedi_rvwatershed = gpd.GeoDataFrame(
    df_gedi_rvwatershed,
    geometry=gpd.points_from_xy(df_gedi_rvwatershed.longitude, df_gedi_rvwatershed.latitude),
    crs="EPSG:4326"
)

print("Performing spatial join...")
gedi_with_county_rvwatershed = gpd.sjoin(
    gdf_gedi_rvwatershed,
    filtered_counties[['jurisdiction', 'geometry']],
    how='inner',
    predicate='within'
)

gedi_county_summary_rvwatershed = (
    gedi_with_county_rvwatershed
    .groupby(['year', 'jurisdiction'])['rh98']
    .mean()
    .reset_index()
    .rename(columns={'rh98': 'canopy_height_mean_m'})
)

print(f"Spatial join complete. Rows in summary: {len(gedi_county_summary_rvwatershed)}")
gedi_county_summary_rvwatershed

Performing spatial join...
Spatial join complete. Rows in summary: 61


,year,jurisdiction,canopy_height_mean_m
0,2019,Albemarle,19.675812
1,2019,Augusta,17.506247
2,2019,Buckingham,15.924260
3,2019,Charlottesville,16.583229
4,2019,Fluvanna,18.198132
...,...,...,...
56,2025,Fluvanna,15.802814
57,2025,Greene,14.136559
58,2025,Louisa,15.712695
59,2025,Nelson,18.998682


In [27]:
# Load dataframe with canopy height data points
df_gedi_rvwatershed_canopy_height = pd.read_csv(OUTPUT_DETAILED_CANOPY_HEIGHT_CSV)

gdf_gedi_rvwatershed_canopy_height = gpd.GeoDataFrame(
     df_gedi_rvwatershed_canopy_height,
     geometry=gpd.points_from_xy(df_gedi_rvwatershed_canopy_height.longitude, df_gedi_rvwatershed_canopy_height.latitude),
     crs="EPSG:4326"
 )

# The Fix for the Value Error

# Drop any stale index columns from a previous join
gdf_gedi_rvwatershed_canopy_height = gdf_gedi_rvwatershed_canopy_height.drop(
    columns=[c for c in ["index_right", "index_left"] if c in gdf_gedi_rvwatershed_canopy_height.columns]
)
filtered_counties = filtered_counties.drop(
    columns=[c for c in ["index_right", "index_left"] if c in filtered_counties.columns]
)

print("Performing spatial join...")
gedi_with_county_canopy_height_rvwatershed = gpd.sjoin(
    gdf_gedi_rvwatershed_canopy_height,
    filtered_counties[['jurisdiction', 'geometry']],
    how='inner',
    predicate='within'
)

print(f"Spatial join complete. Rows in summary: {len(gedi_with_county_canopy_height_rvwatershed)}")
gedi_with_county_canopy_height_rvwatershed.head(20)

Performing spatial join...
Spatial join complete. Rows in summary: 820293


,longitude,latitude,quality_flag,sensitivity,rh98,year,file_source,beam,geometry,index_right,jurisdiction
732,-78.878399,38.093539,1,0.950126,12.68,2019,GEDI02_A_2019152194102_O02655_02_T01408_02_003...,BEAM0000,POINT (-78.8784 38.09354),1,Augusta
733,-78.876905,38.094540,1,0.960131,20.70,2019,GEDI02_A_2019152194102_O02655_02_T01408_02_003...,BEAM0000,POINT (-78.8769 38.09454),1,Augusta
734,-78.876406,38.094873,1,0.906827,2.43,2019,GEDI02_A_2019152194102_O02655_02_T01408_02_003...,BEAM0000,POINT (-78.87641 38.09487),1,Augusta
735,-78.875411,38.095537,1,0.910719,2.39,2019,GEDI02_A_2019152194102_O02655_02_T01408_02_003...,BEAM0000,POINT (-78.87541 38.09554),1,Augusta
736,-78.872920,38.097195,1,0.969524,20.96,2019,GEDI02_A_2019152194102_O02655_02_T01408_02_003...,BEAM0000,POINT (-78.87292 38.09719),1,Augusta
737,-78.861911,38.104453,1,0.903994,2.35,2019,GEDI02_A_2019152194102_O02655_02_T01408_02_003...,BEAM0000,POINT (-78.86191 38.10445),1,Augusta
738,-78.859405,38.106098,1,0.912946,2.43,2019,GEDI02_A_2019152194102_O02655_02_T01408_02_003...,BEAM0000,POINT (-78.8594 38.1061),1,Augusta
739,-78.857401,38.107414,1,0.902428,6.51,2019,GEDI02_A_2019152194102_O02655_02_T01408_02_003...,BEAM0000,POINT (-78.8574 38.10741),1,Augusta
740,-78.856900,38.107743,1,0.903620,3.70,2019,GEDI02_A_2019152194102_O02655_02_T01408_02_003...,BEAM0000,POINT (-78.8569 38.10774),1,Augusta
741,-78.856401,38.108069,1,0.919240,13.77,2019,GEDI02_A_2019152194102_O02655_02_T01408_02_003...,BEAM0000,POINT (-78.8564 38.10807),1,Augusta


In [28]:
#Save to CSV to support County Canopy Height Change
gedi_with_county_canopy_height_rvwatershed.to_csv(OUTPUT_COUNTY_CANOPY_HEIGHT_CSV, index=False)
print(f"Saved locally to : {OUTPUT_COUNTY_CANOPY_HEIGHT_CSV}")

Saved locally to : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_county_canopy_height-rvwatershed.csv


## Save City of Charlottesville details Locally and Upload to S3

In [29]:
# Save locally
gedi_county_summary_rvwatershed.to_csv(OUTPUT_COUNTY_CSV, index=False)
print(f"Saved locally to : {OUTPUT_COUNTY_CSV}")

# Upload to S3 for downstream pipeline consumption
county_csv_filename = os.path.basename(OUTPUT_COUNTY_CSV)
s3_county_key       = GEDI_COUNTY_S3_PREFIX + county_csv_filename
s3_client           = boto3.client("s3")

s3_client.upload_file(
    OUTPUT_COUNTY_CSV,
    S3_BUCKET,
    s3_county_key,
    ExtraArgs={"ContentType": "text/csv"}
)

s3_county_uri = f"s3://{S3_BUCKET}/{s3_county_key}"
print(f"Uploaded to S3   : {s3_county_uri}")
print("\nAll summary phases completed successfully!")

Saved locally to : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_county_summary-rvwatershed.csv
Uploaded to S3   : s3://central-virginia-tree-canopy-project/gedi-county-summary/virginia_gedi_county_summary-rvwatershed.csv

All summary phases completed successfully!


In [30]:
# Save locally
gedi_with_county_canopy_height_rvwatershed.to_csv(OUTPUT_DETAILED_COUNTY_CSV, index=False)
print(f"Saved locally to : {OUTPUT_DETAILED_COUNTY_CSV}")

# Upload to S3 for downstream pipeline consumption
county_csv_filename = os.path.basename(OUTPUT_DETAILED_COUNTY_CSV)
s3_county_key       = GEDI_DETAILED_COUNTY_S3_PREFIX + county_csv_filename
s3_client           = boto3.client("s3")

s3_client.upload_file(
    OUTPUT_DETAILED_COUNTY_CSV,
    S3_BUCKET,
    s3_county_key,
    ExtraArgs={"ContentType": "text/csv"}
)

s3_county_uri = f"s3://{S3_BUCKET}/{s3_county_key}"
print(f"Uploaded to S3   : {s3_county_uri}")
print("\nAll detailed phases completed successfully!")

Saved locally to : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_county_detailed-rvwatershed.csv
Uploaded to S3   : s3://central-virginia-tree-canopy-project/gedi-county-detailed/virginia_gedi_county_detailed-rvwatershed.csv

All detailed phases completed successfully!
